# Test

The test of our proposed algorithm.

## Implementation

copied from the networkx/drawing/layout.py

This is the utility function for the test.

In [2]:
!pip install tqdm

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
import tqdm.auto as tqdm
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx


def make_fig(
    Gs: list, methods: list, file_name: str, legend_pos=1.0, rect_pos=0.96, rows=2
):
    assert len(Gs) % rows == 0

    n2 = len(Gs) // rows
    fig, axes = plt.subplots(
        rows * len(methods), n2, figsize=(3 * n2, 3 * rows * len(methods))
    )
    axes = axes.flatten()

    progress = tqdm.tqdm(total=len(Gs) * len(methods))

    for i, (G, graph_name) in enumerate(Gs):
        for j, (draw_func, _, node_color, kwargs) in enumerate(methods):
            ax = axes[i % n2 + j * n2 + len(methods) * n2 * int(i / n2)]

            t0 = time.perf_counter()
            try:
                pos = draw_func(G, **kwargs)
            except ValueError as e:
                print(f"Error: {e}")
                # kamada_kawai_layout does not support negative edge weights
                pos = np.zeros((len(G), 2))
            t1 = time.perf_counter()

            if type(pos) == np.ndarray:
                nodes = G.nodes()
                pos = dict(zip(nodes, pos))

            if j == 0:
                if graph_name.endswith("_graph"):
                    graph_name = graph_name[:-6]
                if graph_name == "dorogovtsev_goltsev_mendes":
                    graph_name = "DGM"
                ax.set_title(f"{graph_name}\n{t1 - t0:.2f}s", fontsize=20)
            else:
                ax.set_title(f"{t1 - t0:.2f}s", fontsize=20)

            nx.draw(
                G,
                pos=pos,
                ax=ax,
                node_size=min(50, max(5, 5000 // len(G))),
                node_color=node_color,
            )
            ax.axis("on")
            progress.update(1)

    handles = []
    for _, func_name, color, _ in methods:
        handles.append(mpatches.Patch(color=color, label=func_name))

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=len(handles),
        bbox_to_anchor=(0.5, legend_pos),
        fontsize=30,
    )

    plt.tight_layout(rect=[0, 0, 1, rect_pos])
    plt.savefig(file_name)
    plt.close()

## Comparison

### NetworkX Graphs

In [ ]:
def shuffle_nodes(G):
    nodes = list(G.nodes)
    np.random.shuffle(nodes)
    mapping = {node: new_node for node, new_node in zip(G.nodes, nodes)}
    return nx.relabel_nodes(G, mapping)


def graph_generator(n: int):
    graph_functions = [
        (nx.complete_graph, [n]),
        (nx.path_graph, [n]),
        (nx.cycle_graph, [n]),
        (nx.circular_ladder_graph, [n // 2]),
        (nx.wheel_graph, [n]),
        (nx.star_graph, [n]),
        (nx.grid_2d_graph, [int(np.sqrt(n) + 1), int(np.sqrt(n) + 1)]),
        (nx.balanced_tree, [2, int(np.log2(n) + 1)]),
        (nx.binomial_tree, [int(np.log2(n) + 1)]),
        (nx.lollipop_graph, [4 * n // 5, n // 5]),
        (nx.barbell_graph, [2 * n // 5, n // 5]),
        (nx.ladder_graph, [n // 2]),
        (nx.dorogovtsev_goltsev_mendes_graph, [int(np.log(2 * n) / np.log(3) + 1)]),
        (nx.complete_bipartite_graph, [n // 2, n // 2]),
        (nx.barabasi_albert_graph, [n, 2]),
        (nx.powerlaw_cluster_graph, [n, 2, 0.5]),
        (nx.watts_strogatz_graph, [n, 2, 0.5]),
        (nx.random_regular_graph, [3, n]),
        (nx.bipartite.random_graph, [n // 2, n // 2, 0.5]),
        (nx.gnp_random_graph, [n, 0.5]),
        (nx.trivial_graph, []),
    ]
    return [
        (shuffle_nodes(func(*args)), func.__name__) for func, args in graph_functions
    ]


def make_methods(iterations: int):
    return [
        (nx.spring_layout, "FR", "tab:blue", {"iterations": iterations}),
        (
            nx.spring_layout,
            "FR (L-BFGS)",
            "tab:orange",
            {"iterations": iterations, "method": "L-BFGS"},
        ),
        (nx.kamada_kawai_layout, "Kamada--Kawai", "tab:green", {}),
    ]


make_fig(graph_generator(10), make_methods(50), "graphs_10_50", 1.0, 0.96, 3)
make_fig(graph_generator(50), make_methods(50), "graphs_50_50", 1.0, 0.96, 3)
make_fig(graph_generator(500), make_methods(50), "graphs_500_50", 1.0, 0.96, 3)
make_fig(graph_generator(600), make_methods(200), "graphs_600_200", 1.0, 0.96, 3)

  0%|          | 0/63 [00:00<?, ?it/s]

In [16]:
def make_methods_arf_forceatlas(iterations: int):
    return [
        (nx.spring_layout, "FR", "tab:blue", {"iterations": iterations}),
        (
            spring_layout,
            "FR (L-BFGS)",
            "tab:orange",
            {"iterations": iterations, "method": "L-BFGS"},
        ),
        (nx.kamada_kawai_layout, "Kamada--Kawai", "tab:green", {}),
        (nx.arf_layout, "arf", "tab:red", {}),
        (nx.forceatlas2_layout, "ForceAtlas2", "tab:purple", {}),
    ]


make_fig(
    graph_generator(50),
    make_methods_arf_forceatlas(100),
    "arf_forceatlas_50_100",
    1.0,
    0.96,
    3,
)

  0%|          | 0/105 [00:00<?, ?it/s]

### Graphs from SuiteSparse Matrix Collection

In [12]:
import scipy.io
import ssgetpy


def ssgetpy_graphs():
    matrixes = [
        ssgetpy.search(name)
        for name in [
            "can_144",
            "jagmesh1",
            "dwt_1005",
            "1138_bus",
            "bcsstk13",
            "add20",
            "dwt_2680",
            "poli",
            "3elt",
            "USPowerGrid",
            "bcspwr10",
            "memplus",
        ]
    ]

    Gs = []
    for mat in matrixes:
        mat = mat[0]
        path = mat.download(extract=True)[0]
        A = scipy.io.mmread(path + f"/{mat.name}.mtx")
        G = nx.from_scipy_sparse_array(A)
        G.remove_edges_from(nx.selfloop_edges(G))
        Gs.append((G, mat.name))
        print(f"{mat.name}: {len(G)} nodes, {len(G.edges)} edges")
    return Gs


def make_methods_FR(iterations: int):
    return [
        (nx.spring_layout, "FR", "tab:blue", {"iterations": iterations}),
        (
            spring_layout,
            "FR (L-BFGS)",
            "tab:orange",
            {"iterations": iterations, "method": "L-BFGS"},
        ),
    ]


make_fig(ssgetpy_graphs(), make_methods_FR(200), "ssgetpy_17758_200", 1.00, 0.9, 2)

can_144: 144 nodes, 576 edges
jagmesh1: 936 nodes, 2664 edges
dwt_1005: 1005 nodes, 3808 edges
1138_bus: 1138 nodes, 1458 edges
bcsstk13: 2003 nodes, 40940 edges
add20: 2395 nodes, 7462 edges
dwt_2680: 2680 nodes, 11173 edges
poli: 4008 nodes, 4119 edges
3elt: 4720 nodes, 13722 edges
USpowerGrid: 4941 nodes, 6594 edges
bcspwr10: 5300 nodes, 8271 edges
memplus: 17758 nodes, 54196 edges


  0%|          | 0/24 [00:00<?, ?it/s]

### Special Cases

#### Unconnected Graphs

In [ ]:
def separated_graphs():
    def create_and_draw_graphs(n, m):
        graphs = [nx.complete_graph(n) for _ in range(m)]
        G = nx.disjoint_union_all(graphs)
        return G

    Gs = []
    for n, m in [
        (1, 25),
        (5, 5),
        (1, 500),
        (10, 50),
        (250, 2),
        (1, 2000),
        (100, 20),
        (1000, 2),
    ]:
        Gs.append((create_and_draw_graphs(n, m), f"K_{n} * {m}"))
    for n, p in [
        (50, 1e-2),
        (500, 1e-3),
        (500, 5e-3),
        (500, 1e-2),
        (2000, 1e-4),
        (2000, 1e-3),
    ]:
        Gs.append((nx.gnp_random_graph(n, p), f"G({n}, {p})"))

    Gs.append((nx.random_geometric_graph(500, 0.01), "geometric 500"))
    Gs.append((nx.random_geometric_graph(1000, 0.01), "geometric 1000"))

    sizes = [200, 200, 200]
    probs = [[0.25, 0, 0], [0, 0.35, 0], [0, 0, 0.45]]
    Gs.append(
        (
            nx.stochastic_block_model(sizes, probs),
            "stochastic_block",
        )
    )

    Gs.append(
        (nx.disjoint_union(nx.cycle_graph(250), nx.cycle_graph(250)), "C_250 + C_250")
    )

    assert all(not nx.is_connected(G) for G, _ in Gs)

    return Gs


make_fig(separated_graphs(), make_methods(100), "separated_100", 1.00, 0.94, 3)

  0%|          | 0/36 [00:00<?, ?it/s]

#### Very Large Graphs

In [ ]:
# # it takes very very long time, but it works
# G = nx.cycle_graph(100000)
# spring_layout(G, iterations=1, method="L-BFGS")

#### 3D Graphs

In [82]:
G = nx.cycle_graph(50)
pos = spring_layout(G, method="L-BFGS", dim=3)
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.scatter(*zip(*pos.values()))
plt.savefig("3d.png")
plt.close()

#### fixed

In [99]:
G = nx.cycle_graph(40)
pos = spring_layout(
    G,
    method="L-BFGS",
    fixed=[0, 10, 20, 30],
    pos={0: [0, 0], 10: [0, 2], 20: [2, 2], 30: [2, 0]},
)
fig = plt.figure()
nx.draw(G, pos=pos)
plt.savefig("fixed.png")
plt.close()

#### Negative Weights

In [99]:
import matplotlib.pyplot as plt


def unconnected_and_negative(method: str):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    n = 3

    G1 = nx.grid_2d_graph(n, n)
    G2 = nx.grid_2d_graph(n, n)
    G = nx.disjoint_union(G1, G2)
    pos0 = spring_layout(G, iterations=50, method=method)
    pos1 = spring_layout(G, iterations=500, method=method)
    pos2 = spring_layout(G, iterations=5000, method=method)
    nx.draw(G, pos0, ax=axes[0], node_size=1)
    axes[0].set_title("iter:50", fontsize=20)
    nx.draw(G, pos1, ax=axes[1], node_size=1)
    axes[1].set_title("iter:500", fontsize=20)
    nx.draw(G, pos2, ax=axes[2], node_size=1)
    axes[2].set_title("iter:5000", fontsize=20)

    v = G.number_of_nodes()
    G.add_edge(v, 0, weight=-1)
    G.add_edge(v, n**2, weight=-1)
    pos3 = spring_layout(G, iterations=500, method=method)
    nx.draw(G, pos3, ax=axes[3], node_size=1)
    axes[3].set_title("negative weight", fontsize=20)

    fig.suptitle(f"Results by {method}", fontsize=30, y=1.00)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"unconnected_and_negative_{method}.png")
    plt.close()


unconnected_and_negative("FR")
unconnected_and_negative("L-BFGS")